In [1]:
import boto3
import pandas as pd
import os
import pickle
from dotenv import load_dotenv

def load_pkl_from_s3(file_key):
    load_dotenv()
    try:
        s3_client = boto3.client(
            's3',
            aws_access_key_id=os.getenv('NOTEBOOK_ACCESS_KEY'),
            aws_secret_access_key=os.getenv('NOTEBOOK_ACCESS_KEY_SECRET')
        )

        bucket_name = os.getenv('S3_BUCKET_NAME')
        
        obj = s3_client.get_object(Bucket=bucket_name, Key=file_key)
        pkl_data = obj['Body'].read()

        try:
            data = pickle.loads(pkl_data)  # Intentar cargar el objeto desde pickle
        except Exception as e:
            return None


        # Si es un diccionario, intentar convertirlo a DataFrame
        if isinstance(data, dict):
            if 'predictions_labels' in data and isinstance(data['predictions_labels'], (list, tuple, pd.Series)):
                df = pd.DataFrame({'predictions_labels': data['predictions_labels']})
                return df
            else:
                return data

        # Si ya es un DataFrame, regresarlo directamente
        elif isinstance(data, pd.DataFrame):
            return data

        else:
            return None
    except Exception as e:
        return None


In [2]:
path_predictions_score = 'results/predictions_score.pkl'
path_predictions_label = 'results/predictions_label.pkl'

data_score = load_pkl_from_s3(path_predictions_score)
data_labels=load_pkl_from_s3(path_predictions_label)


if 'predictions_score' in data_score and data_score['predictions_score'] is not None:
    date = data_score['date']
    predictions = data_score['predictions_score']

    # Crear el DataFrame, asociando la fecha con cada predicción
    df = pd.DataFrame({
        'date': [date] * len(predictions),  # Repetir la misma fecha para todas las predicciones
        'predictions_score': predictions
    })

else:
    print("⚠️ La clave 'predictions_score' está ausente o tiene un valor None.")
print(type(data_score))
print(type(data_labels))



    
    
  

<class 'dict'>
<class 'pandas.core.frame.DataFrame'>


In [3]:
data_labels.head()

,predictions_labels
0,1
1,1
2,1
3,1
4,1


In [4]:
df.head()


,date,predictions_score
0,2025-02-07 12:26:49.672910,0.755317
1,2025-02-07 12:26:49.672910,0.836077
2,2025-02-07 12:26:49.672910,0.712073
3,2025-02-07 12:26:49.672910,0.766531
4,2025-02-07 12:26:49.672910,0.792728


In [5]:
# Concatenar los dos DataFrames por las filas
concatenated_df = pd.concat([data_labels, df], axis=1)

# Mostrar el DataFrame concatenado
print(concatenated_df.head())


   predictions_labels                       date  predictions_score
0                   1 2025-02-07 12:26:49.672910           0.755317
1                   1 2025-02-07 12:26:49.672910           0.836077
2                   1 2025-02-07 12:26:49.672910           0.712073
3                   1 2025-02-07 12:26:49.672910           0.766531
4                   1 2025-02-07 12:26:49.672910           0.792728
